In [0]:

  CREATE OR REPLACE TABLE workspace.mcphersonsharyn_silver.sea_surface_temperature_silver AS
SELECT
  CAST(latitude_raw  AS DOUBLE)                     AS latitude,
  CAST(longitude_raw AS DOUBLE)                     AS longitude,
  CAST(sst_raw       AS DOUBLE)                     AS sea_surface_temperature,
  source_year                                       AS year,
  source_month                                      AS month,
  'monthly'                                         AS temporal_resolution,
  22                                                AS lat_upper_bound,
  -23                                               AS lat_lower_bound,
  126                                               AS long_upper_bound,
  -73                                               AS long_lower_bound,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    4)                                              AS h3_res4,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    5)                                              AS h3_res5,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    6)                                              AS h3_res6,
  current_timestamp()                               AS processed_timestamp
FROM workspace.mcphersonsharyn_bronze.bronze_sst
WHERE
  latitude_raw       IS NOT NULL
  AND longitude_raw  IS NOT NULL
  AND sst_raw        IS NOT NULL
  AND CAST(sst_raw       AS DOUBLE) != -999
  AND CAST(sst_raw       AS DOUBLE) BETWEEN -2   AND 40
  AND CAST(latitude_raw  AS DOUBLE) BETWEEN -90  AND 90
  AND CAST(longitude_raw AS DOUBLE) BETWEEN -180 AND 180;

In [0]:
CREATE OR REPLACE TABLE workspace.mcphersonsharyn_silver.chlorophyll_silver AS
SELECT
  CAST(latitude_raw  AS DOUBLE)                     AS latitude,
  CAST(longitude_raw AS DOUBLE)                     AS longitude,
  CAST(cc_raw        AS DOUBLE)                     AS chlorophyll_concentration,
  source_year                                       AS year,
  source_month                                      AS month,
  'monthly'                                         AS temporal_resolution,
  22                                                AS lat_upper_bound,
  -23                                               AS lat_lower_bound,
  126                                               AS long_upper_bound,
  -73                                               AS long_lower_bound,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    4)                                              AS h3_res4,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    5)                                              AS h3_res5,
  current_timestamp()                               AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_cc
WHERE
  latitude_raw       IS NOT NULL
  AND longitude_raw  IS NOT NULL
  AND cc_raw         IS NOT NULL
  AND CAST(cc_raw        AS DOUBLE) != -999
  AND CAST(cc_raw        AS DOUBLE) > 0
  AND CAST(cc_raw        AS DOUBLE) < 100
  AND CAST(latitude_raw  AS DOUBLE) BETWEEN -90  AND 90
  AND CAST(longitude_raw AS DOUBLE) BETWEEN -180 AND 180;

In [0]:
SELECT 
  count(*) AS total_rows,
  count(*) filter(WHERE h3_res4 IS NULL) AS null_h3_res4,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3_res5,
  count(*) filter(WHERE h3_res6 IS NULL) AS null_h3_res6,
  count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
  min(sea_surface_temperature) AS min_sst,
  max(sea_surface_temperature) AS max_sst,
  count(DISTINCT month) AS months_loaded
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver;

--new checks after adding h3 - geospacial indexing

In [0]:
SELECT
  count(*) AS total_rows,
  count(*) filter(WHERE h3_res4 IS NULL) AS null_h3_res4,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3_res5,
  count(*) filter(WHERE chlorophyll_concentration IS NULL) AS null_cc,
  min(chlorophyll_concentration) AS min_cc,
  max(chlorophyll_concentration) AS max_cc,
  count(DISTINCT month) AS months_loaded
FROM workspace.mcphersonsharyn_silver.chlorophyll_silver;

--new checks after adding h3 - geospacial indexing

In [0]:
SELECT count(*) FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver;

--new checks after adding h3 - geospacial indexing

In [0]:
SELECT count(*) FROM workspace.mcphersonsharyn_silver.chlorophyll_silver;

--new checks after adding h3 - geospacial indexing

In [0]:
CREATE OR REPLACE TABLE workspace.mcphersonsharyn_silver.environmental_conditions_silver AS
SELECT
  s.latitude,
  s.longitude,
  s.h3_res4,
  s.h3_res5,
  s.h3_res6,
  s.sea_surface_temperature,
  c.chlorophyll_concentration,
  s.year,
  s.month,
  s.temporal_resolution,
  s.lat_upper_bound,
  s.lat_lower_bound,
  s.long_upper_bound,
  s.long_lower_bound,
  current_timestamp()                               AS processed_timestamp
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver s
LEFT JOIN workspace.mcphersonsharyn_silver.chlorophyll_silver c
  ON  s.h3_res4  = c.h3_res4
  AND s.year     = c.year
  AND s.month    = c.month;

In [0]:

SELECT
  count(*) AS total_rows,
  count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
  count(*) filter(WHERE chlorophyll_concentration IS NULL) AS null_cc,
  count(*) filter(WHERE h3_res4 IS NULL) AS null_h3_res4,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3_res5,
  count(DISTINCT month) AS months_loaded,
  (SELECT count(*) FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver) AS sst_silver_count
FROM workspace.mcphersonsharyn_silver.environmental_conditions_silver;

-- new checks after adding h3 - geospacial indexing

In [0]:
SELECT count(*) FROM workspace.mcphersonsharyn_silver.environmental_conditions_silver;

--new checks after adding h3 - geospacial indexing

In [0]:
SELECT
  count(*)                                    AS total_rows,
  count(*) filter(WHERE h3_res5 IS NULL)      AS null_h3,
  count(*) filter(WHERE latitude IS NULL)     AS null_lat,
  count(*) filter(WHERE longitude IS NULL)    AS null_lon,
  count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver;

In [0]:
SELECT
  count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
  count(*) filter(WHERE h3_res5 IS NULL)                 AS null_h3,
  count(*) filter(WHERE latitude IS NULL)                AS null_lat,
  count(*) filter(WHERE longitude IS NULL)               AS null_lon
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver;

In [0]:
SELECT
  latitude_raw,
  longitude_raw,
  CAST(latitude_raw  AS DOUBLE) AS lat_cast,
  CAST(longitude_raw AS DOUBLE) AS lon_cast,
  h3_longlatash3(
    CAST(longitude_raw AS DOUBLE),
    CAST(latitude_raw  AS DOUBLE),
    5) AS h3_test
FROM workspace.mcphersonsharyn_bronze.bronze_sst
LIMIT 10;

In [0]:
-- Check SST Silver
SELECT 'sst_silver' AS table_name,
  count(*) AS total,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver

UNION ALL

-- Check chlorophyll Silver
SELECT 'chlorophyll_silver' AS table_name,
  count(*) AS total,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3
FROM workspace.mcphersonsharyn_silver.chlorophyll_silver

UNION ALL

-- Check unified environmental Silver
SELECT 'environmental_conditions_silver' AS table_name,
  count(*) AS total,
  count(*) filter(WHERE h3_res5 IS NULL) AS null_h3
FROM workspace.mcphersonsharyn_silver.environmental_conditions_silver;

In [0]:
select
  count (*) as null_sst,
  count(*) filter(where chlorophyll_concentration is null) as null_chlorophyll
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
where sea_surface_temperature is null
  or chlorophyll_concentration is null;

In [0]:
select *
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
where sea_surface_temperature < -2
  or sea_surface_temperature > 40
  or chlorophyll_concentration < 0;

In [0]:
select latitude, longitude, year, month, count(*) as duplicate_count
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
group by latitude, longitude, year, month
having count(*) >1;

In [0]:
-- Check for duplicates in chlorophyll Silver
SELECT h3_res5, year, month, count(*) AS duplicate_count
FROM workspace.mcphersonsharyn_silver.chlorophyll_silver
GROUP BY h3_res5, year, month
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;


In [0]:
-- Check for duplicates in SST Silver
SELECT h3_res5, year, month, count(*) AS duplicate_count
FROM workspace.mcphersonsharyn_silver.sea_surface_temperature_silver
GROUP BY h3_res5, year, month
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;

In [0]:
-- Check data types match between tables
DESCRIBE workspace.mcphersonsharyn_silver.sea_surface_temperature_silver;


In [0]:
DESCRIBE workspace.mcphersonsharyn_silver.chlorophyll_silver;


In [0]:
select count(*) as out_of_bounds
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
where latitude> 22  or latitude <23
  or longitude >126 or longitude <-73;

In [0]:
select month, count(*) as num_rows
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
group by month
order by month;

In [0]:
select distinct month
from mcphersonsharyn_silver.silver_environment_clean
order by month;

In [0]:
select *
from mcphersonsharyn_silver.silver_environment_clean
limit 1000;

In [0]:
describe mcphersonsharyn_silver.silver_environment_clean

In [0]:
describe mcphersonsharyn_silver.environmental_conditions_silver


In [0]:

CREATE OR REPLACE VIEW gold_environment_monthly AS
WITH validated AS (
  SELECT
    CAST(latitude AS DOUBLE)                  AS latitude,
    CAST(longitude AS DOUBLE)                 AS longitude,
    h3_res5,
    h3_res6,
    CAST(sea_surface_temperature AS DOUBLE)   AS sst,
    CAST(chlorophyll_concentration AS DOUBLE) AS cc,
    CAST(year AS INTEGER)                     AS year,
    CAST(month AS INTEGER)                    AS month
  FROM workspace.mcphersonsharyn_silver.environmental_conditions_silver
  WHERE
    sea_surface_temperature IS NOT NULL
    AND sea_surface_temperature != -999
    AND sea_surface_temperature BETWEEN -2 AND 40
    AND chlorophyll_concentration IS NOT NULL
    AND chlorophyll_concentration != -999
    AND chlorophyll_concentration > 0
    AND chlorophyll_concentration < 100
    AND latitude BETWEEN -90 AND 90
    AND longitude BETWEEN -180 AND 180
),
deduplicated AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY h3_res5, year, month
      ORDER BY sst DESC
    ) AS rn
  FROM validated
)
SELECT
  latitude,
  longitude,
  h3_res5,
  h3_res6,
  sst,
  cc,
  year,
  month
FROM deduplicated
WHERE rn = 1;

In [0]:
describe gold_environment_monthly;

In [0]:
select * from gold_environment_monthly
limit 100;

In [0]:
select distinct month 
from gold_environment_monthly
order by month;

In [0]:
select year, month, count(*) as row_count
from mcphersonsharyn_silver.environmental_conditions_silver
group by year, month
order by year, month;

In [0]:
select year, month, count(*) as row_count
from mcphersonsharyn_silver.environmental_conditions_silver
where sea_surface_temperature is not null
  and chlorophyll_concentration is not null
  and sea_surface_temperature between -2 and 40
  and chlorophyll_concentration >= 0
  and latitude between -90 and 90
  and longitude between -180 and 180
group by year, month
order by year, month;

In [0]:
with silver_counts as (
SELECT
year,
month,
count(*) as silver_rows
FROM mcphersonsharyn_silver.environmental_conditions_silver
group by year, month
),
gold_counts as (
SELECT
year,
month,
count(*) as gold_rows
FROM gold_environment_monthly
group by year, month
)
select
s.year,
s.month,
s.silver_rows,
coalesce(g.gold_rows, 0) as gold_rows,
s.silver_rows - coalesce(g.gold_rows, 0) as rows_removed
from silver_counts s
left join gold_counts g
on s.year = g.year
and s.month = g.month
order by s.year, s.month;


In [0]:
select
  count(*) as total_rows,
  sum(case when sea_surface_temperature is null then 1 else 0 end) as null_sst,
  sum(case when chlorophyll_concentration is null then 1 else 0 end) as null_cc, 
  sum(case when sea_surface_temperature not between -2 and 40 then 1 else 0 end) as invalid_sst,
  sum(case when chlorophyll_concentration < 0 then 1 else 0 end) as invalid_cc,
  sum(case when latitude not between -90 and 90 then 1 else 0 end) as invalid_lat,
  sum(case when longitude not between -180 and 180 then 1 else 0 end) as invalid_long
from workspace.mcphersonsharyn_silver.environmental_conditions_silver
where month = 11;